# Baseline Model Training

This notebook will be used to train and evaluete initial machine-learning models for the Board Game Rating Predictor project.

Goal is to create simple baseline models that predict `Rating Average` using the prepared modelling datasets created during feature engineering.

This notebook will:

- load prepared training and testing datasets
- train simple baseline model
- train initial regression model
- evaluate model performance
- compare model results
- document baseline modelling decisions

In [224]:
# Import libraries used throghout this notebook
# numpy needed to calculate RMSE by taking the square root of MSE
import numpy as np
import pandas as pd

# Import Path for creating file paths that work
# reliably across different operating systems.
from pathlib import Path

# Import baseline modelling toool
from sklearn.dummy import DummyRegressor
# LinearRegression -first real regression model we are training
from sklearn.linear_model import LinearRegression

# Import model evaluation metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [225]:
# Define current notebook folder and project root
current_folder = Path.cwd()
project_root = current_folder.parent

print("Current folder;", current_folder)
print("Project root:", project_root)

Current folder; c:\Users\Zombie\Desktop\vscode-projects\board-game-rating-predictor\notebooks
Project root: c:\Users\Zombie\Desktop\vscode-projects\board-game-rating-predictor


## Load Prepared Modelling Datasets

The prepared training and testing datasets are loaded from the `data/processed` folder.

These files were created during the feature engineering notebook.

Loaded files are following:
- training features
- testing features
- training target values
- testing target values
- model feature names
- imputation summary

Because the feature files were already prepared and imputed, they should not contain missing values.

In [226]:
# Define paths to prepared modelling dataset files
processed_data_folder = project_root / "data" / "processed"

x_train_path = processed_data_folder / "x_train_prepared.csv"
x_test_path = processed_data_folder / "x_test_prepared.csv"
y_train_path = processed_data_folder / "y_train.csv"
y_test_path = processed_data_folder / "y_test.csv"

model_feature_names_path = processed_data_folder / "model_feature_names.csv"
imputation_summary_path = processed_data_folder / "imputation_summary.csv"

print("Processed data folder:", processed_data_folder)
print("Folder exists:", processed_data_folder.exists())

Processed data folder: c:\Users\Zombie\Desktop\vscode-projects\board-game-rating-predictor\data\processed
Folder exists: True


In [227]:
# Cofrim that all required prepared modelling files exist
prepared_file_paths = [
    x_train_path,
    x_test_path,
    y_train_path,
    y_test_path,
    model_feature_names_path,
    imputation_summary_path
]

for file_path in prepared_file_paths:
    print(file_path.name, "exists:", file_path.exists())

x_train_prepared.csv exists: True
x_test_prepared.csv exists: True
y_train.csv exists: True
y_test.csv exists: True
model_feature_names.csv exists: True
imputation_summary.csv exists: True


In [228]:
# Load prepared modelling datasets
X_train = pd.read_csv(x_train_path)
X_test = pd.read_csv(x_test_path)

y_train_data = pd.read_csv(y_train_path)
y_test_data = pd.read_csv(y_test_path)

model_feature_names = pd.read_csv(model_feature_names_path)
imputation_summary = pd.read_csv(imputation_summary_path)

print("Prepared modelling files loaded successfully.")

Prepared modelling files loaded successfully.


In [229]:
# Define target colum
target_column = "Rating Average"

# Convert target DataFrames into Series for modelling
y_train = y_train_data[target_column]
y_test = y_test_data[target_column]

print("y_train type:", type(y_train))
print("y_test type:", type(y_test))

y_train type: <class 'pandas.Series'>
y_test type: <class 'pandas.Series'>


In [230]:
# Check the shapes of the loaded modelling datasets
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("model_feature_names shape:", model_feature_names.shape)
print("imputation_summary shape:", imputation_summary.shape)

X_train shape: (16273, 35)
X_test shape: (4069, 35)
y_train shape: (16273,)
y_test shape: (4069,)
model_feature_names shape: (35, 1)
imputation_summary shape: (6, 2)


In [231]:
# Verify if loaded feature columns match saved modl feature names
expected_feature_names = model_feature_names["Feature"].tolist()

x_train_columns_match = X_train.columns.tolist() == expected_feature_names
x_test_columns_match = X_test.columns.tolist() == expected_feature_names

print("X_train columns match saved feature names:", x_train_columns_match)
print("X_test columns match saved feature names:", x_test_columns_match)

X_train columns match saved feature names: True
X_test columns match saved feature names: True


In [232]:
# Check missing values in the loaded prepared datasets
print("Missing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in X_test:", X_test.isnull().sum().sum())
print("Missing values in y_train:", y_train.isnull().sum())
print("Missing values in y_test:", y_test.isnull().sum())

Missing values in X_train: 0
Missing values in X_test: 0
Missing values in y_train: 0
Missing values in y_test: 0


In [233]:
# Preview loaded training features
X_train.head()

,Year Published,Min Players,Min Age,Complexity Average,Max Players Log,Play Time Log,Year Published Missing,Min Players Missing,Max Players Missing,Play Time Missing,...,Mechanic: Area Movement,Mechanic: Simultaneous Action Selection,Domain: Wargames,Domain: Strategy Games,Domain: Family Games,Domain: Thematic Games,Domain: Abstract Games,Domain: Children's Games,Domain: Party Games,Domain: Customizable Games
0,2017.0,2.0,10.0,2.00,1.791759,4.110874,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2013.0,2.0,11.0,1.43,1.945910,3.828641,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2019.0,2.0,8.0,2.00,1.098612,3.433987,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2004.0,2.0,12.0,3.00,1.098612,7.090910,0,0,0,0,...,1,0,1,0,0,0,0,0,0,0
4,2010.0,2.0,8.0,1.40,1.945910,2.772589,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [234]:
# Preview the loaded training target values
y_train.head()

0    5.86
1    4.53
2    7.33
3    7.60
4    5.97
Name: Rating Average, dtype: float64

### Prepared Modelling Dataset Load Result

Prepared modelling datasets were loaded successfully from the `data/processed` folder.

Training feature dataset contains 16,273 rows and 35 feature columns.

Testing feature dataset contains 4,069 rows and 35 feature columns.

Target values were loaded and converted into one-dimensional Series objects for modelling.

Loaded feature columns match the saved model feature names.

Loaded prepared datasets contain no missing values and and ready for baseline model training.

## Train Baseline Dummy Model

A baseline dummy model is trained before building more advanced machine-learning models.

This baseline model uses the mean `Rating Average` value from the training data and predicts that same value for every game.

Dummy model is not accurate (but it is not expected to be). Its purpose is to provide a simple comparison point.

Future models should perform better than this baseline.

In [235]:
# Create a baseline dummy regression model
baseline_dummy_model = DummyRegressor(strategy="mean")

# Train the dummy model on the training data
baseline_dummy_model.fit(X_train, y_train)

print("Baseline dummy model trained successfully.")
print("Baseline startegy:", baseline_dummy_model.strategy)

Baseline dummy model trained successfully.
Baseline startegy: mean


In [236]:
# Dummy model predicts training target mean for every row
baseline_prediction_value = baseline_dummy_model.constant_.item()

print("Training target mean:", y_train.mean())
print("Baseline model constant prediction:", baseline_prediction_value)

Training target mean: 6.403027714619308
Baseline model constant prediction: 6.403027714619308


In [237]:
# Create baseline predictions for training and testing feature sets
baseline_train_predictions = baseline_dummy_model.predict(X_train)
baseline_test_predictions = baseline_dummy_model.predict(X_test)

print("Baseline train predictions shape:", baseline_train_predictions.shape)
print("Baseline test predictions shape:", baseline_test_predictions.shape)

Baseline train predictions shape: (16273,)
Baseline test predictions shape: (4069,)


In [238]:
# Check how many unique prediction values the dummy model has created
unique_baseline_train_predictions = pd.Series(
    baseline_train_predictions
).nunique()

unique_baseline_test_predictions = pd.Series(
    baseline_test_predictions
).nunique()

print("Unique baseline train prediction values:", unique_baseline_train_predictions)
print("Unique baseline test prediction values:", unique_baseline_test_predictions)

Unique baseline train prediction values: 1
Unique baseline test prediction values: 1


In [239]:
# Preview actual testing ratings beside baseline predictions
baseline_prediction_preview = pd.DataFrame(
    {
        "Actual Rating": y_test.head(10).values,
        "Baseline Prediction": baseline_test_predictions[:10]
    }
)

baseline_prediction_preview

,Actual Rating,Baseline Prediction
0,6.23,6.403028
1,7.02,6.403028
2,6.96,6.403028
3,7.07,6.403028
4,8.88,6.403028
5,7.68,6.403028
6,6.88,6.403028
7,6.17,6.403028
8,7.53,6.403028
9,6.15,6.403028


### Baseline Dummy Model Result

A baseline dummy regression model was trained using training dataset.

The model uses the mean `Rating Average` value from training target values.

It predicts the same rating value for every board game.

Model is purposly made simple and will be used as a comparison point for future regression models.

A useful machine-learning model should perform better than this dummy baseline.

## Train Linear Regression Model

A Linear Regression model is trained as the first real machine-learning model.

Unlike the dummy baseline model we created before, Linear Regression uses the prepared feature columns to make predictions.

The purpose of this model is to check whether a simple regression model can learn useful patterns from the board game features and perform better than the dummy baseline.

In [240]:
# Create Linear Regression model
linear_regression_model = LinearRegression()

# Train model using the prepared training features and training target
linear_regression_model.fit(X_train, y_train)

print("Linear Regression model trained successfully (Party time!!!)")

Linear Regression model trained successfully (Party time!!!)


In [241]:
# Check the trained Linear Regression model structure
# Intercept is the model’s starting prediction before feature effects added
print("Model intercept:", linear_regression_model.intercept_)
#  Number of coefficients should match the number of feature columns
print("Number of model coefficients:", len(linear_regression_model.coef_))
print("Number of training features:", X_train.shape[1])

Model intercept: 4.63285114944448
Number of model coefficients: 35
Number of training features: 35


In [242]:
# This creates predictions that will be evaluate later. 
# We create predictions for both training and tetsing data
# We later compare how the mode performs on seen VS unseen data
linear_train_predictions = linear_regression_model.predict(X_train)
linear_test_predictions = linear_regression_model.predict(X_test)

print("Linear Regression train predictions shape:", linear_train_predictions.shape)
print("Linear Regression test predictions shape:", linear_test_predictions.shape)

Linear Regression train predictions shape: (16273,)
Linear Regression test predictions shape: (4069,)


In [243]:
# Review the distribution of Linear Regression test predictions
# - helps to quickly check wheather the predictions look reasonable
linear_test_prediction_summary = pd.Series(
    linear_test_predictions,
    name="Linear Regression Test Predictions"
).describe()

linear_test_prediction_summary

count    4069.000000
mean        6.401083
std         0.538451
min         4.198142
25%         6.025324
50%         6.394544
75%         6.765077
max         8.116889
Name: Linear Regression Test Predictions, dtype: float64

In [244]:
# Preview actual testing ratings beside Linear Regression predictions
# -> human-readable first look at predictions
linear_prediction_preview = pd.DataFrame(
    {
        "Actual Rating": y_test.head(10).values,
        "Linear Regression Prediction": linear_test_predictions[:10]
    }
)

linear_prediction_preview

,Actual Rating,Linear Regression Prediction
0,6.23,6.370959
1,7.02,6.276297
2,6.96,7.099881
3,7.07,6.450589
4,8.88,7.706117
5,7.68,6.606565
6,6.88,7.144250
7,6.17,6.573564
8,7.53,6.496619
9,6.15,6.002259


### Linear Regression Model Result

A Lineart Regression model was trained using the prepared training dataset.

The model used all 35 prepared feature columns to learn relationships between board game attributes and `Rating Average`.

Predictions were created for both the training and the testing datasets.

We are not ealuating model predictions against dummy baseline just yet (in next step)

## Evaluate Baseline Model Performance

Dummy baseline model and Linear Regression model are evalutaed using regression metrics.

The metrics used are:

- Mean Absolute Error, or MAE
- Root Mean Squared Error, or RMSE
- R-squared, or R2 (R square)

MAE and RMSE measure prediction error, so lower values are better.

R2 (R square) measures how much target variation the model explains, so bigger values are better.

In [245]:
# Create reuseable function for evaluating regression model performance
# that way it can be reusead for ebvery model I train
def evaluate_regression_model(y_true, y_pred):
    """Calculate common regression evaluation metrics."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    return {
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": float(r2)
    }

In [246]:
# For showing how dummy baseline model performs
baseline_train_metrics = evaluate_regression_model(
    y_train,
    baseline_train_predictions
)

baseline_test_metrics = evaluate_regression_model(
    y_test,
    baseline_test_predictions
)

print("Baseline train metrics:", baseline_train_metrics)
print("Baseline test metrics:", baseline_test_metrics)

Baseline train metrics: {'MAE': 0.7322148819090196, 'RMSE': 0.9349506830454742, 'R2': 0.0}
Baseline test metrics: {'MAE': 0.7278226238025776, 'RMSE': 0.9397375154091072, 'R2': -1.189410751889497e-06}


In [247]:
# Evaluate Linear Regression model on training and testing data
# - this uses same metrics on Linear Regression predictions 
# - if Linear Regression has lower MAE/RMSE and higher R2 then dummy baseline on test set
linear_train_metrics = evaluate_regression_model(
    y_train,
    linear_train_predictions
)

linear_test_metrics = evaluate_regression_model(
    y_test,
    linear_test_predictions
)

print("Linear Regression train metrics:", linear_train_metrics)
print("Linear Regression test metrics:", linear_test_metrics)

Linear Regression train metrics: {'MAE': 0.5784354054738692, 'RMSE': 0.7548808071189349, 'R2': 0.34810243229366933}
Linear Regression test metrics: {'MAE': 0.5707408416338806, 'RMSE': 0.750716202249207, 'R2': 0.3618262076147285}


In [248]:
# Combine model performance metrics.
# table that put all results in one place 
# for easier compare between dummy model and Linear Regression
model_performance = pd.DataFrame(
    [
        {
            "Model": "Dummy Baseline",
            "Dataset": "Train",
            **baseline_train_metrics
        },
        {
            "Model": "Dummy Baseline",
            "Dataset": "Test",
            **baseline_test_metrics
        },
        {
            "Model": "Linear Regression",
            "Dataset": "Train",
            **linear_train_metrics
        },
        {
            "Model": "Linear Regression",
            "Dataset": "Test",
            **linear_test_metrics
        }
    ]
)

model_performance

,Model,Dataset,MAE,RMSE,R2
0,Dummy Baseline,Train,0.732215,0.934951,0.000000
1,Dummy Baseline,Test,0.727823,0.939738,-0.000001
2,Linear Regression,Train,0.578435,0.754881,0.348102
3,Linear Regression,Test,0.570741,0.750716,0.361826


In [249]:
# Create rounded version of the model performance table to easy read
model_performance_rounded = model_performance.copy()

model_performance_rounded[["MAE", "RMSE", "R2"]] = (
    model_performance_rounded[["MAE", "RMSE", "R2"]]
    .round(4)
)

model_performance_rounded
# full precision table is good for exact values
# rounded numbers is easier to read in notebook 
# both versions are kept 

,Model,Dataset,MAE,RMSE,R2
0,Dummy Baseline,Train,0.7322,0.9350,0.0000
1,Dummy Baseline,Test,0.7278,0.9397,-0.0000
2,Linear Regression,Train,0.5784,0.7549,0.3481
3,Linear Regression,Test,0.5707,0.7507,0.3618


In [250]:
# - EArly Check if models are learnign actually something useful 
# Calculate Linear Regression improvement over dummy baseline on test set
mae_improvement = (
    baseline_test_metrics["MAE"]
    - linear_test_metrics["MAE"]
)

rmse_improvement = (
    baseline_test_metrics["RMSE"]
    - linear_test_metrics["RMSE"]
)

print("Test MAE improvement:", mae_improvement)
print("Test RMSE improvement:", rmse_improvement)

Test MAE improvement: 0.15708178216869695
Test RMSE improvement: 0.18902131315990023


### Baseline Model Evaluation Result

Dummy baseline model and Linear Regression model were evaluated using MAE, RMSE, and R2 (R square)

Dummy baseline gives simple comparison point because they predict training target mean for every game.

Linear Regression model uses prepared feature colums to create different predictions for different games.

The test-set metrics will be used in the next step to compare the initial model results more clearly.

## Compare Initial Model Results

The dummy baseline model and Linear Regression model are compared using their test-set performance.

The test set is the most important comparison because it contains data the models did not see during training.

A useful model should have lower MAE and RMSE than the dummy baseline, and a higher R2 (r square) score.

In [251]:
# This creates smaller table using only test-set results
# -> easier to read than full train/test table when deciding which model is better
test_model_comparison = pd.DataFrame(
    [
        {
            "Model": "Dummy Baseline",
            "Test MAE": baseline_test_metrics["MAE"],
            "Test RMSE": baseline_test_metrics["RMSE"],
            "Test R2": baseline_test_metrics["R2"]
        },
        {
            "Model": "Linear Regression",
            "Test MAE": linear_test_metrics["MAE"],
            "Test RMSE": linear_test_metrics["RMSE"],
            "Test R2": linear_test_metrics["R2"]
        }
    ]
)

test_model_comparison

,Model,Test MAE,Test RMSE,Test R2
0,Dummy Baseline,0.727823,0.939738,-0.000001
1,Linear Regression,0.570741,0.750716,0.361826


In [252]:
# exact table is better for calculations
# rounded table is better for interpretation
test_model_comparison_rounded = test_model_comparison.copy()

test_model_comparison_rounded[["Test MAE", "Test RMSE", "Test R2"]] = (
    test_model_comparison_rounded[["Test MAE", "Test RMSE", "Test R2"]]
    .round(4)
)

test_model_comparison_rounded

,Model,Test MAE,Test RMSE,Test R2
0,Dummy Baseline,0.7278,0.9397,-0.0000
1,Linear Regression,0.5707,0.7507,0.3618


In [253]:
# showing how better Linear Regression is in practical error term
mae_error_reduction = (
    baseline_test_metrics["MAE"]
    - linear_test_metrics["MAE"]
)

rmse_error_reduction = (
    baseline_test_metrics["RMSE"]
    - linear_test_metrics["RMSE"]
)

mae_error_reduction_percentage = (
    mae_error_reduction
    / baseline_test_metrics["MAE"]
    * 100
)

rmse_error_reduction_percentage = (
    rmse_error_reduction
    / baseline_test_metrics["RMSE"]
    * 100
)

print("Test MAE error reduction:", mae_error_reduction)
print("Test MAE error reduction percentage:", mae_error_reduction_percentage)
print("Test RMSE error reduction:", rmse_error_reduction)
print("Test RMSE error reduction percentage:", rmse_error_reduction_percentage)

Test MAE error reduction: 0.15708178216869695
Test MAE error reduction percentage: 21.582426408787413
Test RMSE error reduction: 0.18902131315990023
Test RMSE error reduction percentage: 20.114267022489926


In [254]:
# 1 Create table
# 2 turns improvement numbers into clean format 
# -> to show how much Linear Regression is better
error_reduction_summary = pd.DataFrame(
    [
        {
            "Metric": "MAE",
            "Dummy Baseline Test Error": baseline_test_metrics["MAE"],
            "Linear Regression Test Error": linear_test_metrics["MAE"],
            "Absolute Error Reduction": mae_error_reduction,
            "Percentage Error Reduction": mae_error_reduction_percentage
        },
        {
            "Metric": "RMSE",
            "Dummy Baseline Test Error": baseline_test_metrics["RMSE"],
            "Linear Regression Test Error": linear_test_metrics["RMSE"],
            "Absolute Error Reduction": rmse_error_reduction,
            "Percentage Error Reduction": rmse_error_reduction_percentage
        }
    ]
)

error_reduction_summary_rounded = error_reduction_summary.copy()

error_reduction_summary_rounded[
    [
        "Dummy Baseline Test Error",
        "Linear Regression Test Error",
        "Absolute Error Reduction",
        "Percentage Error Reduction"
    ]
] = (
    error_reduction_summary_rounded[
        [
            "Dummy Baseline Test Error",
            "Linear Regression Test Error",
            "Absolute Error Reduction",
            "Percentage Error Reduction"
        ]
    ]
    .round(4)
)

error_reduction_summary_rounded

,Metric,Dummy Baseline Test Error,Linear Regression Test Error,Absolute Error Reduction,Percentage Error Reduction
0,MAE,0.7278,0.5707,0.1571,21.5824
1,RMSE,0.9397,0.7507,0.1890,20.1143


In [255]:
# Identify better model according to each test-set metirc
best_model_by_mae = test_model_comparison.loc[
    test_model_comparison["Test MAE"].idxmin(),
    "Model"
]

best_model_by_rmse = test_model_comparison.loc[
    test_model_comparison["Test RMSE"].idxmin(),
    "Model"
]

best_model_by_r2 = test_model_comparison.loc[
    test_model_comparison["Test R2"].idxmax(),
    "Model"
]

print("Best model by Test MAE:", best_model_by_mae)
print("Best model by Test RMSE:", best_model_by_rmse)
print("Best model by Test R2:", best_model_by_r2)

Best model by Test MAE: Linear Regression
Best model by Test RMSE: Linear Regression
Best model by Test R2: Linear Regression


In [256]:
# Create short conclusion based on initial model comparison
initial_model_comparison_conclusion = (
    "Linear Regression performed better than the dummy baseline on the test set. "
    f"It reduced MAE by {mae_error_reduction:.4f} "
    f"({mae_error_reduction_percentage:.2f}%) and reduced RMSE by "
    f"{rmse_error_reduction:.4f} ({rmse_error_reduction_percentage:.2f}%). "
    f"The Linear Regression test R2 score was {linear_test_metrics['R2']:.4f}."
)

initial_model_comparison_conclusion

'Linear Regression performed better than the dummy baseline on the test set. It reduced MAE by 0.1571 (21.58%) and reduced RMSE by 0.1890 (20.11%). The Linear Regression test R2 score was 0.3618.'

### Initial Model Comparison Result

The Linear Regression model performed better than the dummy baseline on the test set.

Linear Regression had lower MAE and RMSE than the dummy baseline, meaning its predictions had smaller average errors.

Linear Regression also had a higher R2 (R square) score, meaning it explained more variation in board game ratings than the dummy baseline.

This confirms that the prepared features contain useful predictive information.

Linear Regression is now the best initial model, but future modelling steps may try more flexible models to improve performance further.

## Document Baseline Modelling Results

This section documents the main results from the initial baseline modelling work.

The dummy baseline model and Linear Regression model are summarized using their test-set performance.

The purpose is to clearly record which model performed better, how much improvement was achieved, and what this means for the next modelling steps.

In [257]:
# Structured way of interpretation (table)
# Diff from matric table is explanation in 'plain0 langugae
baseline_modelling_results_summary = pd.DataFrame(
    [
        {
            "Result Area": "Best initial model",
            "Summary": best_model_by_mae,
            "Explanation": (
                "Linear Regression had the lowest test MAE, lowest test RMSE, "
                "and highest test R2 among the initial models."
            )
        },
        {
            "Result Area": "Dummy baseline purpose",
            "Summary": "Mean rating prediction",
            "Explanation": (
                "The dummy baseline predicts the training-set average rating "
                "for every board game and provides a minimum comparison point."
            )
        },
        {
            "Result Area": "Linear Regression test MAE",
            "Summary": round(linear_test_metrics["MAE"], 4),
            "Explanation": (
                "On average, Linear Regression predictions were about this many "
                "rating points away from the true test ratings."
            )
        },
        {
            "Result Area": "Linear Regression test RMSE",
            "Summary": round(linear_test_metrics["RMSE"], 4),
            "Explanation": (
                "RMSE gives stronger penalty to larger errors, so this helps show "
                "whether the model makes large mistakes."
            )
        },
        {
            "Result Area": "Linear Regression test R2",
            "Summary": round(linear_test_metrics["R2"], 4),
            "Explanation": (
                "The Linear Regression model explained part of the variation in "
                "test-set board game ratings."
            )
        },
        {
            "Result Area": "MAE improvement over baseline",
            "Summary": f"{mae_error_reduction_percentage:.2f}%",
            "Explanation": (
                "Linear Regression reduced test MAE compared with the dummy baseline."
            )
        },
        {
            "Result Area": "RMSE improvement over baseline",
            "Summary": f"{rmse_error_reduction_percentage:.2f}%",
            "Explanation": (
                "Linear Regression reduced test RMSE compared with the dummy baseline."
            )
        },
        {
            "Result Area": "Next modelling direction",
            "Summary": "Try more flexible models",
            "Explanation": (
                "Linear Regression beat the baseline, but future models may capture "
                "non-linear relationships and improve performance further."
            )
        }
    ]
)

print(
    "Baseline modelling results summary shape:",
    baseline_modelling_results_summary.shape
)

baseline_modelling_results_summary

Baseline modelling results summary shape: (8, 3)


,Result Area,Summary,Explanation
0,Best initial model,Linear Regression,"Linear Regression had the lowest test MAE, low..."
1,Dummy baseline purpose,Mean rating prediction,The dummy baseline predicts the training-set a...
2,Linear Regression test MAE,0.5707,"On average, Linear Regression predictions were..."
3,Linear Regression test RMSE,0.7507,"RMSE gives stronger penalty to larger errors, ..."
4,Linear Regression test R2,0.3618,The Linear Regression model explained part of ...
5,MAE improvement over baseline,21.58%,Linear Regression reduced test MAE compared wi...
6,RMSE improvement over baseline,20.11%,Linear Regression reduced test RMSE compared w...
7,Next modelling direction,Try more flexible models,"Linear Regression beat the baseline, but futur..."


In [258]:
# Define output paths for baseline modelling result files
baseline_model_performance_path = (
    processed_data_folder / "baseline_model_performance.csv"
)

initial_model_comparison_path = (
    processed_data_folder / "initial_model_comparison.csv"
)

error_reduction_summary_path = (
    processed_data_folder / "initial_model_error_reduction_summary.csv"
)

baseline_modelling_results_summary_path = (
    processed_data_folder / "baseline_modelling_results_summary.csv"
)

# Save baseline modelling result files
model_performance.to_csv(baseline_model_performance_path, index=False)
test_model_comparison.to_csv(initial_model_comparison_path, index=False)
error_reduction_summary.to_csv(error_reduction_summary_path, index=False)
baseline_modelling_results_summary.to_csv(
    baseline_modelling_results_summary_path,
    index=False
)

print("Baseline modelling result files saved successfully.")

Baseline modelling result files saved successfully.


In [259]:
# SAFETY CHECK - Confrim that baseline modelling result files were created
# (notebook did not only create tables in memory, but actually in project)
baseline_result_file_paths = [
    baseline_model_performance_path,
    initial_model_comparison_path,
    error_reduction_summary_path,
    baseline_modelling_results_summary_path
]

for file_path in baseline_result_file_paths:
    print(file_path.name, "exists:", file_path.exists())

baseline_model_performance.csv exists: True
initial_model_comparison.csv exists: True
initial_model_error_reduction_summary.csv exists: True
baseline_modelling_results_summary.csv exists: True


In [260]:
# Reload saved basleine modelling summary-
# -to verify saved CSV can be resued later
baseline_modelling_results_summary_check = pd.read_csv(
    baseline_modelling_results_summary_path
)

print(
    "Baseline modelling results summary check shape:",
    baseline_modelling_results_summary_check.shape
)

baseline_modelling_results_summary_check

Baseline modelling results summary check shape: (8, 3)


,Result Area,Summary,Explanation
0,Best initial model,Linear Regression,"Linear Regression had the lowest test MAE, low..."
1,Dummy baseline purpose,Mean rating prediction,The dummy baseline predicts the training-set a...
2,Linear Regression test MAE,0.5707,"On average, Linear Regression predictions were..."
3,Linear Regression test RMSE,0.7507,"RMSE gives stronger penalty to larger errors, ..."
4,Linear Regression test R2,0.3618,The Linear Regression model explained part of ...
5,MAE improvement over baseline,21.58%,Linear Regression reduced test MAE compared wi...
6,RMSE improvement over baseline,20.11%,Linear Regression reduced test RMSE compared w...
7,Next modelling direction,Try more flexible models,"Linear Regression beat the baseline, but futur..."


### Baseline Modelling Result

This notebook loaded the prepared modelling datasets and trained two initial models.

The dummy baseline model predicted the training-set average `Rating Average` for every board game.

The Linear Regression model used all 35 prepared feature columns to make different predictions for different games.

Linear Regression performed better than the dummy baseline on the test set.

It reduced test MAE by approximately 21.58% and reduced test RMSE by approximately 20.11% compared with the dummy baseline.

The Linear Regression test R2 (R square) score was approximately 0.3618.

This confirms that the prepared board game features contain useful predictive information.

Linear Regression is the best initial model so far, but future modelling work can try more flexible models to improve performance further.